In [24]:
# 1. Install Python Packages (LlamaIndex, ChromaDB, Streamlit)
!pip install -q llama-index-core \
                llama-index-llms-ollama \
                llama-index-embeddings-huggingface \
                llama-index-vector-stores-chroma \
                llama-index-readers-file \
                chromadb \
                streamlit

# 2. Install Linux System Tools (pciutils for GPU detection)
!apt-get update && apt-get install -y pciutils > /dev/null

# 3. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n✅ Installation Complete! System is ready.")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
###########################################################

In [83]:
import subprocess
import time
import os

# 1. Kill any existing ollama to start fresh
!pkill -9 ollama
time.sleep(2)

# 2. Start the server (using DEVNULL to keep the screen clean)
print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Wait 20 seconds (Server load aaga time edukum)
print("Waiting for server to initialize (20s)...")
time.sleep(20)

# 4. Pull the model (Explicitly run and wait for it)
print("Pulling Qwen 2.5 Model...")
!ollama pull qwen2.5:1.5b

# 5. Final check to see if it's actually there
print("\n--- Verification ---")
check = os.popen("ollama list").read()
if "qwen2.5:1.5b" in check:
    print(" SUCCESS: Qwen 2.5 is loaded and ready!")
else:
    print("ERROR: Model not found. Check your internet connection.")

Starting Ollama server...
Waiting for server to initialize (20s)...
Pulling Qwen 2.5 Model...


--- Verification ---
 SUCCESS: Qwen 2.5 is loaded and ready!


In [80]:
!curl http://127.0.0.1:11434

Ollama is running

In [63]:
%%writefile backend_logic.py
import os
import shutil
import chromadb
import uuid
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

def init_settings():
    # Models setup
    Settings.llm = Ollama(model="qwen2.5:1.5b", base_url="http://127.0.0.1:11434", request_timeout=600.0)
    Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

def process_documents(data_path="./data"):
    init_settings()

    # 1. ALWAYS start with a fresh folder name to kill old memory
    unique_id = str(uuid.uuid4())[:8]
    db_path = f"./chroma_db_{unique_id}"

    # 2. Setup fresh ChromaDB
    db = chromadb.PersistentClient(path=db_path)
    chroma_collection = db.get_or_create_collection("rag_collection")
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

    # 3. Load ONLY current files from the folder
    if not os.path.exists(data_path): os.makedirs(data_path)
    documents = SimpleDirectoryReader(data_path).load_data()

    if not documents:
        raise ValueError("No documents found in folder!")

    # 4. Build fresh index
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )
    return index

Overwriting backend_logic.py


In [32]:
!pip install streamlit

In [84]:
%%writefile app.py
import streamlit as st
import os
from backend_logic import process_documents, init_settings
from llama_index.core.prompts import PromptTemplate

# Basic Page Config
st.set_page_config(page_title="Qwen AI", layout="wide")

st.title("Qwen 🩵 AI Assistant")

if 'initialized' not in st.session_state:
    init_settings()
    st.session_state.initialized = True

# Sidebar for Upload and Indexing
with st.sidebar:
    st.header("Upload Document")
    uploaded_file = st.file_uploader("Choose a PDF or TXT file", type=['pdf', 'txt'])

    if uploaded_file:
        if not os.path.exists("./data"):
            os.makedirs("./data")

        # Clear previous files to avoid mixing data
        for f in os.listdir("./data"):
            os.remove(os.path.join("./data", f))

        file_path = os.path.join("./data", uploaded_file.name)
        with open(file_path, "wb") as f:
            f.write(uploaded_file.getbuffer())
        st.success(f"File '{uploaded_file.name}' saved.")

    if st.button("Build Knowledge Base"):
        with st.spinner("Processing..."):
            st.session_state.index = process_documents()
            st.success("Knowledge base ready.")

# Chat History Display
if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Chat Input and Logic
if prompt := st.chat_input("Enter your question..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    if "index" in st.session_state:
        with st.chat_message("assistant"):
            with st.spinner("Searching..."):
                # Strict Prompt Logic
                qa_prompt_str = (
                    "Context information is below.\n"
                    "---------------------\n"
                    "{context_str}\n"
                    "---------------------\n"
                    "Answer ONLY using the provided context. If the answer is NOT in the context, "
                    "strictly say: 'I am sorry, but this information is not in the uploaded document.'\n"
                    "Query: {query_str}\n"
                    "Answer: "
                )

                qa_template = PromptTemplate(qa_prompt_str)
                query_engine = st.session_state.index.as_query_engine(
                    streaming=True,
                    text_qa_template=qa_template
                )

                response = query_engine.query(prompt)

            # Response Streaming
            placeholder = st.empty()
            full_response = ""
            for text in response.response_gen:
                full_response += text
                placeholder.markdown(full_response + "▌")
            placeholder.markdown(full_response)

            st.session_state.messages.append({"role": "assistant", "content": full_response})
    else:
        st.warning("Please upload a document and build the knowledge base first.")

Overwriting app.py


In [85]:
import os, time, subprocess, requests

# Cleanup
!pkill -9 streamlit
!pkill -9 cloudflared

# Download cloudflared if needed
if not os.path.exists("cloudflared"):
    !wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Start Streamlit
print("Starting Streamlit...")
os.system("nohup python3 -m streamlit run app.py --server.port 8501 --server.address 127.0.0.1 --server.headless true > streamlit.log 2>&1 &")

# Ping to ensure it's up
for i in range(15):
    try:
        if requests.get("http://127.0.0.1:8501").status_code == 200:
            print("\n✅Streamlit is UP!")
            break
    except:
        print(".", end=""); time.sleep(5)

# Tunnel
print("\n🔗 Click the link below:")
!./cloudflared tunnel --url http://127.0.0.1:8501

Starting Streamlit...
.
✅Streamlit is UP!

🔗 Click the link below:
2026-03-23T09:02:46Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-23T09:02:46Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-23T09:02:50Z INF +--------------------------------------------------------------------------------------------+
2026-03-23T09:02:50Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-23T09: